## Prompt Optimization with MLflow

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4 langchain==1.3.14 langgraph==1.2.10 mlflow==3.15.2 dspy==3.3.1

  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
   ---------------------------------------- 0.0/23.4 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/23.4 MB 11.9 MB/s eta 0:00:02
   -------- ------------------------------- 4.7/23.4 MB 11.2 MB/s eta 0:00:02
   --------- ------------------------------ 5.5/23.4 MB 9.0 MB/s eta 0:00:02
   ---------- ----------------------------- 6.3/23.4 MB 7.7 MB/s eta 0:00:03
   ----------- ---------------------------- 6.8/23.4 MB 6.4 MB/s eta 0:00:03
   ------------ --------------------------- 7.6/23.4 MB 6.0 MB/s eta 0:00:03
   --------------- ------------------------ 8.9/23.4 MB 6.1 MB/s eta 0:00:03
   ----------------- ---------------------- 10.5/23.4 MB 6.2 MB/s eta 0:00:03
   -------------------- ------------------- 12.1/23.4 MB 6.3 MB/s eta 0:00:02
   ----------------------- ---------------- 13.6/

  You can safely remove it manually.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting up the Environment

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("CLAUDE_API_KEY")
anthropic_model_name = os.getenv("CLAUDE_MODEL_NAME")

os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key

### Instantiating the ChatAnthropic Class

In [2]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Enable MLflow Tracing

In [3]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("prompt-optimization")
mlflow.set_tracking_uri("http://localhost:5000")

c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/01 23:42:26 INFO mlflow.tracking.fluent: Experiment with name 'prompt-optimization' does not exist. Creating a new experiment.


### Register the Initial Prompt

In [4]:
PROMPT_NAME = "ticket-classification-prompt"

prompt = mlflow.genai.register_prompt(
    name=PROMPT_NAME,

    template="""
Classify the following IT support ticket.

Ticket:
{{ticket}}
""",

    commit_message="v1: Basic ticket classification prompt"
)


# set a production alias
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="development",
    version=1
)

print(f"Prompt: {prompt.name}")
print(f"Version: {prompt.version}")

2026/09/01 23:42:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: ticket-classification-prompt, version 1


Prompt: ticket-classification-prompt
Version: 1


### Define the Prediction Function

In [5]:
from langchain_core.messages import HumanMessage

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}@development"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        response = model.invoke(
                [HumanMessage(content=formatted_prompt)]
        )

        return response.content

### Test your Prediction Function

In [6]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)


print(output)

## Ticket Classification

| Field | Value |
|-------|-------|
| **Category** | Hardware |
| **Subcategory** | Monitor/Display |
| **Priority** | Low |
| **Urgency** | Low |
| **Impact** | Low |

## Reasoning

- **Low priority**: The issue is intermittent ("occasionally") and does not prevent the user from working ("can continue working normally")
- **No immediate action required**: This is a non-blocking issue that can be scheduled for routine troubleshooting
- **Potential causes**: Loose cable connection, refresh rate settings, graphics driver, or early signs of hardware degradation

## Suggested Initial Response

Schedule routine troubleshooting to check display cables, connection ports, refresh rate settings, and graphics drivers.


### Optimize against Data

In [7]:
# Training data with inputs, expected outputs, and expectations
dataset = [
    {
        "inputs": {
            "query": (
                "I changed my password this morning and "
                "now I cannot log into my corporate account."
            )
        },
        "outputs": {
            "response": "Category: Authentication\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Authentication'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "The office Wi-Fi is unavailable for everyone "
                "on the third floor and nobody can access "
                "internal applications."
            )
        },
        "outputs": {
            "response": "Category: Network\nPriority: Critical"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Network'",
                "Priority must be 'Critical'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "Microsoft Excel crashes whenever I try to "
                "open one particular spreadsheet. Other "
                "spreadsheets work normally."
            )
        },
        "outputs": {
            "response": "Category: Software\nPriority: Medium"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Software'",
                "Priority must be 'Medium'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "I received an email asking me to enter my "
                "company password on an unfamiliar website."
            )
        },
        "outputs": {
            "response": "Category: Security\nPriority: High"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Security'",
                "Priority must be 'High'"
            ]
        }
    },

    {
        "inputs": {
            "query": (
                "My second monitor occasionally flickers, "
                "but I can continue working normally."
            )
        },
        "outputs": {
            "response": "Category: Hardware\nPriority: Low"
        },
        "expectations": {
            "expected_facts": [
                "Category must be 'Hardware'",
                "Priority must be 'Low'"
            ]
        }
    }
]

In [8]:
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

# Optimize the prompt
result = mlflow.genai.optimize_prompts(
    predict_fn = predict_fn,
    train_data = dataset,
    prompt_uris = [prompt.uri],
    optimizer = GepaPromptOptimizer(reflection_model = "anthropic:/claude-opus-4-5"),
    scorers = [Correctness(model="anthropic:/claude-opus-4-5")]
)

# Use the Optimized Prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized Template: {optimized_prompt.template}")

2026/09/01 23:43:00 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/09/01 23:43:01 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


🏃 View run silent-pug-325 at: http://localhost:5000/#/experiments/1/runs/bf9e64199f9e4ea79b110529d43347a0
🧪 View experiment at: http://localhost:5000/#/experiments/1


ImportError: GEPA >= 0.0.26 is required. Please install it with: `pip install 'gepa>=0.0.26'`

Trace(trace_id=tr-20f3873fdd92e2ee98db8290f3ed467d)

### Use the Optimized Prompt

In [ ]:
VERSION_NUMBER = "LATEST_PROMPT_VERSION_NUMBER"

def predict_fn(query):

        prompt = mlflow.genai.load_prompt(
            name_or_uri=(
                f"prompts:/{PROMPT_NAME}/{VERSION_NUMBER}"
            )
        )

        formatted_prompt = prompt.format(
            ticket=query
        )

        response = model.invoke(
                [HumanMessage(content=formatted_prompt)]
        )

        return response.content

In [ ]:
ticket_query = "My second monitor occasionally flickers, but I can continue working normally"

output = predict_fn(query = ticket_query)


print(output)